In [1]:
import torch
import torch.nn as nn
import math

print(torch.cuda.is_available())

False


In [ ]:
class InputEmbeddings(nn.Module):

    def __init__(self,d_model: int , vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self,x:torch.Tensor)->torch.Tensor:
        return self.embedding(x) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
    



class PostionalEncoding(nn.Module):
    def __init__(self,d_model: int , seq_len: int , dropout: float)->None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # Create a matrix of shape (seq_len,d_model)

        Postionalencoding = torch.zeros(seq_len , d_model)
        
        # Create a vector of shape (seq_len,1)

        position = torch.arange(0,seq_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model , 2).float()*(-torch.log(10000.0)/d_model))


        # Apply the sin to even and cos to odd postions

        position[:,0::2] = torch.sin(position*div_term)
        position[:,1::2] = torch.cos(position*div_term)

        position = position.unsqueeze(0)  # for batch

        self.register_buffer('postion',position)


    def forward(self,x):
        x = x + (self.position[:, :x.shape[1],:]).requires_grad_(False)

        return self.dropout(x)    



In [ ]:
class LayerNormalization(nn.Module):

    def __init__(self,eps: float = 10**-6)->None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(1))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self,x):
        mean = x.mean(dim = -1 , keepdim = True)
        std = x.std(dim = -1 , keepdim = True)
        return self.alpha*(x-mean) / (std + self.eps) + self.bias



class FeedForwardBlock(nn.Module):

    def __init__(self,d_model: int , d_ff: int , dropout: float)->None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model , d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff,d_model)

    def forward(self,x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))




class MultiheadAttensionBlock(nn.Module):
    def __init__(self,d_model: int , h: int , dropout: float)->None:

        super().__init__()
        self.d_model = d_model
        self.h = h

        assert d_model % h == 0 , "d_model is not divisible by h"

        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model,d_model) # Query
        self.w_k = nn.Linear(d_model,d_model) # Key
        self.w_v = nn.Linear(d_model,d_model) # Value
        self.w_o  = nn.Linear(d_model,d_model) # final
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attension(query , key , value , mask , dropout: nn.Dropout):

        d_k = query.shape[-1]

        #(Batch , h , seq_Len , d_k) --> (Batch , h , seq_len , seq_len)

        attension_score = (query @ key.transpose(-2,-1)) / math.sqrt(d_k)

        if mask is not None:
            attension_score.masked_fill_(mask == 0 , -1e9)
        attension_score = attension_score.softmax(dim = -1) # (Batch,h,seq_len,seq_len)

        if dropout is not None:
            attension_score = dropout(attension_score)

        return (attension_score @ value) , attension_score


    def forward(self, q , k, v , mask):
        query = self.w_q(q) # (Batch,seq_len,d_model)-->(Batch,seq_len,d_model)
        key = self.w_k(k)   # (Batch,seq_len,d_model)-->(Batch,seq_len,d_model)     
        value = self.w_v(v) # (Batch,seq_len,d_model)-->(Batch,seq_len,d_model)

        # (Batch , seq_len , d_model)-->(Batch,seq_len,h,d_k)-->(Batch,h,seq_len,d_k)
        query = query.view(query.shape[0],query.shape[1], self.h,self.d_k).transpose(1,2)
        key = key.view(key.shape[0],key.shape[1], self.h,self.d_k).transpose(1,2)
        value = value.view(value.shape[0],value.shape[1], self.h,self.d_k).transpose(1,2)

       
        x, self.attension_score = MultiheadAttensionBlock.attension(query,key,value,mask,self.dropout)

        # (Batch,h,seq_len, d_k)-->(Batch,seq_len ,h,d_k)-->(Batch,seq_len,d_model)
        x = x.transpose(1,2).contiguous().view(x.shape[0],-1,self.h*self.d_k)
        #(Batch , seq_len,d_model)-->(Batch,seq_len,d_model)
        return self.w_o(x)
    



class ResidualConnection(nn.Module):

    def __init__(self,dropout: float)->None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization()

    def forward(self, x , sublayer):
        return x + self.dropout(sublayer(self.norm(x)))
        


class EncoderBlock(nn.Module):
    def __init__(self,self_attension_block: MultiheadAttensionBlock , feed_forward_block: FeedForwardBlock , dropout: float)->None:
        super().__init__()
        self.self_attension_block = self_attension_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])


    """"
      x  → original input, added back at end (skip connection)
      sublayer  → the lambda function (attention wrapped in one arg)

      1. self.norm(x)          → normalize x
      2. sublayer(norm(x))     → pass into attention(norm(x), norm(x), norm(x), mask)
      3. self.dropout(...)     → apply dropout
      4. x + ...               → add original x back
    """


    def forward(self,x,src_mask):
        x = self.residual_connections[0](x,lambda x: self.self_attension_block(x,x,x,src_mask))
        x = self.residual_connections[1](x,self.feed_forward_block)
        return x
        


class Encoder(nn.Module):

    def __init__(self,layers: nn.ModuleList)->None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self,x,mask):
        for layer in self.layers:
            x = layer(x,mask)

        return self.norm(x)



In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, features: int, self_attention_block: MultiheadAttensionBlock, cross_attention_block: MultiheadAttensionBlock, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, self.feed_forward_block)
        return x
    
class Decoder(nn.Module):

    def __init__(self, features: int, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

class ProjectionLayer(nn.Module):

    def __init__(self, d_model, vocab_size) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x) -> None:
        # (batch, seq_len, d_model) --> (batch, seq_len, vocab_size)
        return self.proj(x)
    
class Transformer(nn.Module):

    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PostionalEncoding, tgt_pos: PostionalEncoding, projection_layer: ProjectionLayer) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        # (batch, seq_len, d_model)
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)
    
    def decode(self, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor):
        # (batch, seq_len, d_model)
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def project(self, x):
        # (batch, seq_len, vocab_size)
        return self.projection_layer(x)

In [ ]:
def build_transformer(src_vocab_size: int, tgt_vocab_size: int, src_seq_len: int, tgt_seq_len: int, d_model: int=512, N: int=6, h: int=8, dropout: float=0.1, d_ff: int=2048)-> Transformer:
    # Create the embedding layers
    src_embed = InputEmbeddings(d_model,src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

    # Create the positional encoding layers

    src_pos = PostionalEncoding(d_model, src_seq_len , dropout)
    tgt_pos = PostionalEncoding(d_model, tgt_seq_len , dropout)

    # Create the encoder blocks
    encoder_blocks = []
    for _ in range(N):
        encoder_self_attension_block = MultiheadAttensionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model,d_ff,dropout)
        encoder_block = EncoderBlock(encoder_self_attension_block, feed_forward_block , dropout)
        encoder_blocks.append(encoder_block)

    # Create the decoder blocks
    decoder_blocks = []
    for _ in range(N):
        decoder_self_attention_block = MultiheadAttensionBlock(d_model,h, dropout)
        decoder_cross_attension_block = MultiheadAttensionBlock(d_model, h, dropout)
        feed_forward_block = FeedForwardBlock(d_model, d_ff , dropout)
        decoder_block = DecoderBlock(decoder_self_attention_block, decoder_cross_attension_block, feed_forward_block, dropout)    
        decoder_blocks.append(decoder_block)

    # Create the encoder and the decoder
    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))

    # Create the projection layer

    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)

    # Create the transformer

    transformer = Transformer(encoder, decoder , src_embed , tgt_embed , src_pos , tgt_pos , projection_layer)    

